# 18-fixed — Full rerun: post-group-stage exact knockout forecast

Run this notebook from the repository root after the World Cup group stage is complete.

It recomputes everything from the packaged inputs:

- exact group-stage results;
- final group standings;
- third-place ranking;
- official Round of 32 bracket using FIFA Annex C;
- post-group team strength recalibration;
- knockout-stage probabilities;
- one coherent projected knockout path;
- charts and a report bundle.

This is a forecasting and simulation project, not betting advice.


In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    """
    Find the repository root regardless of whether the notebook is launched
    from the repo root or from the notebooks/ folder.
    """
    start = Path.cwd() if start is None else Path(start)
    start = start.resolve()

    required = Path("data/examples/18_post_group_stage_knockout/group_stage_results_full.csv")

    candidates = [start] + list(start.parents)

    for candidate in candidates:
        if (candidate / required).exists():
            return candidate

    checked = "\n".join(str(candidate / required) for candidate in candidates)

    raise FileNotFoundError(
        "Could not find the repository root. Checked these paths:\n"
        + checked
    )


REPO_ROOT = find_repo_root()

print("Current working directory:", Path.cwd())
print("Detected repository root:", REPO_ROOT)

INPUT_DIR = REPO_ROOT / "data/examples/18_post_group_stage_knockout"
OUTPUT_DIR = REPO_ROOT / "data/processed/18_post_group_stage_exact_knockout_forecast"
FIGURE_DIR = REPO_ROOT / "assets/figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = INPUT_DIR / "group_stage_results_full.csv"
ANNEX_C_PATH = INPUT_DIR / "official_annex_c_2026.csv"
TEAM_STRENGTH_PATH = INPUT_DIR / "updated_team_strength_after_group_stage.csv"

if not RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing {RESULTS_PATH}")

if not ANNEX_C_PATH.exists():
    raise FileNotFoundError(f"Missing {ANNEX_C_PATH}")

if not TEAM_STRENGTH_PATH.exists():
    raise FileNotFoundError(f"Missing {TEAM_STRENGTH_PATH}")

results_df = pd.read_csv(RESULTS_PATH)
annex_df = pd.read_csv(ANNEX_C_PATH)
team_strength_input = pd.read_csv(TEAM_STRENGTH_PATH)

print("Loaded group-stage results:", len(results_df))
print("Loaded Annex C rows:", len(annex_df))
print("Loaded team strength rows:", len(team_strength_input))


In [ ]:
GROUPS = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Switzerland", "Canada", "Bosnia & Herzegovina", "Qatar"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Paraguay", "Turkey"],
    "E": ["Germany", "Côte d'Ivoire", "Ecuador", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Uruguay", "Saudi Arabia"],
    "I": ["France", "Norway", "Senegal", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Colombia", "Portugal", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

TEAM_TO_GROUP = {
    team: group
    for group, teams in GROUPS.items()
    for team in teams
}

teams = sorted(TEAM_TO_GROUP.keys())

# The final official group order after the completed group stage.
# This is used as the authoritative ranking order when tie-breakers are not
# fully represented by points/GD/GF alone.
OFFICIAL_GROUP_ORDER = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Switzerland", "Canada", "Bosnia & Herzegovina", "Qatar"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["United States", "Australia", "Paraguay", "Turkey"],
    "E": ["Germany", "Côte d'Ivoire", "Ecuador", "Curaçao"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Uruguay", "Saudi Arabia"],
    "I": ["France", "Norway", "Senegal", "Iraq"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Colombia", "Portugal", "DR Congo", "Uzbekistan"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

OFFICIAL_RANK = {
    team: i + 1
    for group, order in OFFICIAL_GROUP_ORDER.items()
    for i, team in enumerate(order)
}


In [ ]:
def compute_group_standings(results):
    stats = {
        team: {
            "team": team,
            "group": TEAM_TO_GROUP[team],
            "played": 0,
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "goal_difference": 0,
            "points": 0,
        }
        for team in teams
    }

    for _, row in results.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        hg = int(row["home_goals"])
        ag = int(row["away_goals"])

        for team, gf, ga in [(home, hg, ag), (away, ag, hg)]:
            item = stats[team]
            item["played"] += 1
            item["goals_for"] += gf
            item["goals_against"] += ga
            item["goal_difference"] += gf - ga

            if gf > ga:
                item["wins"] += 1
                item["points"] += 3
            elif gf == ga:
                item["draws"] += 1
                item["points"] += 1
            else:
                item["losses"] += 1

    standings = pd.DataFrame(stats.values())
    standings["official_group_rank"] = (
        standings["team"].map(OFFICIAL_RANK).astype(int)
    )

    standings = standings.sort_values(
        ["group", "official_group_rank"]
    ).reset_index(drop=True)

    return standings


standings = compute_group_standings(results_df)

assert len(results_df) == 72
assert standings["played"].sum() == 144
assert len(annex_df) == 495
assert all(
    ranks == [1, 2, 3, 4]
    for ranks in standings.groupby("group")["official_group_rank"].apply(list)
)

standings.to_csv(OUTPUT_DIR / "final_group_standings.csv", index=False)

display(standings)


In [ ]:
top_two = standings[standings["official_group_rank"] <= 2].copy()
thirds = standings[standings["official_group_rank"] == 3].copy()

thirds_ranked = thirds.sort_values(
    ["points", "goal_difference", "goals_for"],
    ascending=[False, False, False],
).reset_index(drop=True)

thirds_ranked["third_place_rank"] = np.arange(1, len(thirds_ranked) + 1)

best_thirds = thirds_ranked.head(8).copy()

qualified = pd.concat(
    [top_two, best_thirds.drop(columns=["third_place_rank"])],
    ignore_index=True,
)

qualified["qualified_via"] = np.where(
    qualified["official_group_rank"] <= 2,
    "top_two",
    "best_third",
)

third_combo_key = "".join(sorted(best_thirds["group"]))

annex_row = annex_df[annex_df["combo_key"] == third_combo_key]

if len(annex_row) != 1:
    raise ValueError(
        f"Expected exactly one Annex C row for {third_combo_key}, "
        f"found {len(annex_row)}"
    )

annex_row = annex_row.iloc[0]

thirds_ranked.to_csv(OUTPUT_DIR / "third_place_ranking.csv", index=False)
qualified.to_csv(OUTPUT_DIR / "qualified_teams.csv", index=False)
pd.DataFrame([annex_row]).to_csv(OUTPUT_DIR / "actual_annex_c_row.csv", index=False)

print("Third-place combo:", third_combo_key)
print("Annex C option:", int(annex_row["option"]))
display(thirds_ranked)
display(qualified)


In [ ]:
R32_FIXED = {
    "M73": ("2A", "2B"),
    "M74": ("1E", "3ABCDF"),
    "M75": ("1F", "2C"),
    "M76": ("1C", "2F"),
    "M77": ("1I", "3CDFGH"),
    "M78": ("2E", "2I"),
    "M79": ("1A", "3CEFHI"),
    "M80": ("1L", "3EHIJK"),
    "M81": ("1D", "3BEFIJ"),
    "M82": ("1G", "3AEHIJ"),
    "M83": ("2K", "2L"),
    "M84": ("1H", "2J"),
    "M85": ("1B", "3EFGIJ"),
    "M86": ("1J", "2H"),
    "M87": ("1K", "3DEIJL"),
    "M88": ("2D", "2G"),
}

R16_PAIRS = [
    ("M89", "M74", "M77"),
    ("M90", "M73", "M75"),
    ("M91", "M76", "M78"),
    ("M92", "M79", "M80"),
    ("M93", "M83", "M84"),
    ("M94", "M81", "M82"),
    ("M95", "M86", "M88"),
    ("M96", "M85", "M87"),
]

QF_PAIRS = [
    ("M97", "M89", "M90"),
    ("M98", "M93", "M94"),
    ("M99", "M91", "M92"),
    ("M100", "M95", "M96"),
]

SF_PAIRS = [
    ("M101", "M97", "M98"),
    ("M102", "M99", "M100"),
]

FINAL_PAIR = ("M104", "M101", "M102")

lookup = {}

for _, row in standings.iterrows():
    rank = int(row["official_group_rank"])
    group = row["group"]
    lookup[f"{rank}{group}"] = row["team"]


def team_for_slot(slot, match_id):
    if not slot.startswith("3"):
        return lookup[slot]

    group_winner_slot = R32_FIXED[match_id][0]
    third_source = annex_row[group_winner_slot]

    return lookup[third_source]


r32_rows = []

for match_id, (left_slot, right_slot) in R32_FIXED.items():
    r32_rows.append({
        "match_id": match_id,
        "stage": "round_of_32",
        "slot_a": left_slot,
        "slot_b": right_slot,
        "team_a": team_for_slot(left_slot, match_id),
        "team_b": team_for_slot(right_slot, match_id),
    })

round_of_32 = pd.DataFrame(r32_rows)

unique_r32_teams = len(
    set(round_of_32[["team_a", "team_b"]].values.ravel())
)

if unique_r32_teams != 32:
    raise ValueError(f"Expected 32 unique R32 teams, got {unique_r32_teams}")

round_of_32.to_csv(OUTPUT_DIR / "official_round_of_32_bracket.csv", index=False)

display(round_of_32)


In [ ]:
def expected_score(elo_a, elo_b):
    return 1.0 / (1.0 + 10 ** (-(elo_a - elo_b) / 400.0))


def margin_multiplier(goal_diff, elo_diff):
    margin = max(abs(goal_diff), 1)
    return np.log(margin + 1.0) * (2.2 / (0.001 * abs(elo_diff) + 2.2))


strength = team_strength_input.copy()

if "starting_elo" in strength.columns:
    initial_elo = dict(zip(strength["team"], strength["starting_elo"]))
elif "post_group_elo" in strength.columns:
    # Fallback for older packages.
    initial_elo = dict(zip(strength["team"], strength["post_group_elo"]))
else:
    initial_elo = {team: 1500.0 for team in teams}

elo = {
    team: float(initial_elo.get(team, 1500.0))
    for team in teams
}

starting_elo = elo.copy()

K = 32.0

for _, row in results_df.sort_values("date").iterrows():
    home = row["home_team"]
    away = row["away_team"]
    hg = int(row["home_goals"])
    ag = int(row["away_goals"])

    actual_home = 1.0 if hg > ag else 0.5 if hg == ag else 0.0
    expected_home = expected_score(elo[home], elo[away])

    delta = (
        K
        * margin_multiplier(hg - ag, elo[home] - elo[away])
        * (actual_home - expected_home)
    )

    elo[home] += delta
    elo[away] -= delta

updated_strength = pd.DataFrame([
    {
        "team": team,
        "group": TEAM_TO_GROUP[team],
        "starting_elo": starting_elo[team],
        "post_group_elo": elo[team],
        "elo_change": elo[team] - starting_elo[team],
        "qualified": team in set(qualified["team"]),
    }
    for team in teams
]).sort_values("post_group_elo", ascending=False)

updated_strength.to_csv(
    OUTPUT_DIR / "updated_team_strength_after_group_stage.csv",
    index=False,
)

display(updated_strength.head(20))


In [ ]:
def p_win(team_a, team_b):
    return expected_score(elo[team_a], elo[team_b])


def combine_match(left_distribution, right_distribution):
    out = defaultdict(float)

    for team_a, prob_a_reaches in left_distribution.items():
        for team_b, prob_b_reaches in right_distribution.items():
            pair_probability = prob_a_reaches * prob_b_reaches
            p_a = p_win(team_a, team_b)

            out[team_a] += pair_probability * p_a
            out[team_b] += pair_probability * (1.0 - p_a)

    return dict(out)


match_distributions = {}
match_participant_distributions = {}

for _, row in round_of_32.iterrows():
    match_id = row["match_id"]
    team_a = row["team_a"]
    team_b = row["team_b"]

    left = {team_a: 1.0}
    right = {team_b: 1.0}

    match_participant_distributions[match_id] = (left, right)
    match_distributions[match_id] = combine_match(left, right)

for pairs in [R16_PAIRS, QF_PAIRS, SF_PAIRS, [FINAL_PAIR]]:
    for match_id, prev_a, prev_b in pairs:
        left = match_distributions[prev_a]
        right = match_distributions[prev_b]

        match_participant_distributions[match_id] = (left, right)
        match_distributions[match_id] = combine_match(left, right)


stage_probabilities = {team: defaultdict(float) for team in teams}

for team in set(qualified["team"]):
    stage_probabilities[team]["round_of_32"] = 1.0

for match_id in R32_FIXED:
    for team, probability in match_distributions[match_id].items():
        stage_probabilities[team]["round_of_16"] += probability

for match_id, _, _ in R16_PAIRS:
    for team, probability in match_distributions[match_id].items():
        stage_probabilities[team]["quarterfinal"] += probability

for match_id, _, _ in QF_PAIRS:
    for team, probability in match_distributions[match_id].items():
        stage_probabilities[team]["semifinal"] += probability

for match_id, _, _ in SF_PAIRS:
    for team, probability in match_distributions[match_id].items():
        stage_probabilities[team]["final"] += probability

for team, probability in match_distributions["M104"].items():
    stage_probabilities[team]["champion"] = probability

probability_rows = []

for team in sorted(set(qualified["team"])):
    probability_rows.append({
        "team": team,
        "group": TEAM_TO_GROUP[team],
        "post_group_elo": elo[team],
        "round_of_32_probability": stage_probabilities[team]["round_of_32"],
        "round_of_16_probability": stage_probabilities[team]["round_of_16"],
        "quarterfinal_probability": stage_probabilities[team]["quarterfinal"],
        "semifinal_probability": stage_probabilities[team]["semifinal"],
        "final_probability": stage_probabilities[team]["final"],
        "champion_probability": stage_probabilities[team]["champion"],
    })

knockout_probabilities = pd.DataFrame(probability_rows)

knockout_probabilities = knockout_probabilities.sort_values(
    "champion_probability",
    ascending=False,
).reset_index(drop=True)

knockout_probabilities.to_csv(
    OUTPUT_DIR / "knockout_stage_probabilities.csv",
    index=False,
)

display(knockout_probabilities.head(15))


In [ ]:
def deterministic_winner(team_a, team_b):
    return team_a if p_win(team_a, team_b) >= 0.5 else team_b


deterministic_participants = {}
deterministic_winners = {}

for _, row in round_of_32.iterrows():
    match_id = row["match_id"]
    team_a = row["team_a"]
    team_b = row["team_b"]

    deterministic_participants[match_id] = (team_a, team_b)
    deterministic_winners[match_id] = deterministic_winner(team_a, team_b)

for pairs in [R16_PAIRS, QF_PAIRS, SF_PAIRS, [FINAL_PAIR]]:
    for match_id, prev_a, prev_b in pairs:
        team_a = deterministic_winners[prev_a]
        team_b = deterministic_winners[prev_b]

        deterministic_participants[match_id] = (team_a, team_b)
        deterministic_winners[match_id] = deterministic_winner(team_a, team_b)


def stage_for_match(match_id):
    if match_id in R32_FIXED:
        return "round_of_32"

    if any(match_id == item[0] for item in R16_PAIRS):
        return "round_of_16"

    if any(match_id == item[0] for item in QF_PAIRS):
        return "quarterfinal"

    if any(match_id == item[0] for item in SF_PAIRS):
        return "semifinal"

    return "final"


single_bracket_rows = []

for match_id, (team_a, team_b) in sorted(
    deterministic_participants.items(),
    key=lambda item: int(item[0].replace("M", "")),
):
    single_bracket_rows.append({
        "match_id": match_id,
        "stage": stage_for_match(match_id),
        "team_a": team_a,
        "team_b": team_b,
        "p_team_a": p_win(team_a, team_b),
        "p_team_b": 1.0 - p_win(team_a, team_b),
        "projected_winner": deterministic_winners[match_id],
    })

single_bracket = pd.DataFrame(single_bracket_rows)

single_bracket.to_csv(
    OUTPUT_DIR / "single_projected_knockout_bracket.csv",
    index=False,
)

projected_champion = deterministic_winners["M104"]

display(single_bracket)
print("Projected champion:", projected_champion)


In [ ]:
summary = {
    "stage": "post_group_stage",
    "group_stage_complete": True,
    "group_stage_results_loaded": int(len(results_df)),
    "qualified_teams": int(len(qualified)),
    "annex_c_rows": int(len(annex_df)),
    "third_combo_key": third_combo_key,
    "annex_c_option": int(annex_row["option"]),
    "round_of_32_unique_teams": int(unique_r32_teams),
    "probability_method": (
        "exact_dynamic_programming_over_fixed_knockout_bracket"
    ),
    "strength_method": (
        "post_group_elo_update_from_all_72_group_matches"
    ),
    "top_champion_by_probability": knockout_probabilities.iloc[0]["team"],
    "top_champion_probability": float(
        knockout_probabilities.iloc[0]["champion_probability"]
    ),
    "projected_champion_single_bracket": projected_champion,
}

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Copy core inputs/outputs to the processed output folder.
results_df.to_csv(OUTPUT_DIR / "group_stage_results_full.csv", index=False)
standings.to_csv(OUTPUT_DIR / "final_group_standings.csv", index=False)
thirds_ranked.to_csv(OUTPUT_DIR / "third_place_ranking.csv", index=False)
qualified.to_csv(OUTPUT_DIR / "qualified_teams.csv", index=False)
annex_df.to_csv(OUTPUT_DIR / "official_annex_c_2026.csv", index=False)

print(json.dumps(summary, indent=2))


In [ ]:
top = knockout_probabilities.head(12)

plt.figure(figsize=(10, 7))
plt.barh(
    top["team"][::-1],
    top["champion_probability"][::-1],
)
plt.xlabel("Champion probability")
plt.title("Post-group-stage champion probabilities")
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "18_post_group_stage_champion_probabilities.png",
    dpi=180,
)
plt.show()

plt.figure(figsize=(11, 9))
plt.axis("off")
plt.title("Official Round of 32 after group stage", fontsize=16)

for i, row in round_of_32.iterrows():
    y = 0.96 - i * 0.057
    label = f"{row['match_id']}: {row['team_a']} vs {row['team_b']}"
    plt.text(0.02, y, label, fontsize=10)

plt.text(
    0.02,
    0.02,
    f"Annex C option {int(annex_row['option'])} | "
    f"third-place groups: {third_combo_key}",
    fontsize=9,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "18_official_round_of_32_bracket.png",
    dpi=180,
)
plt.show()

plt.figure(figsize=(14, 10))
plt.axis("off")
plt.title("Projected knockout path after group stage", fontsize=16)

stage_order = [
    "round_of_32",
    "round_of_16",
    "quarterfinal",
    "semifinal",
    "final",
]

for x, stage in enumerate(stage_order):
    part = single_bracket[
        single_bracket["stage"] == stage
    ].reset_index(drop=True)

    plt.text(
        x / 4,
        0.98,
        stage.replace("_", " ").title(),
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

    n = len(part)

    for i, row in part.iterrows():
        y = 0.92 - (i + 0.5) * (0.82 / max(n, 1))
        text = (
            f"{row['match_id']}\n"
            f"{row['team_a']} ({row['p_team_a']:.0%})\n"
            f"{row['team_b']} ({row['p_team_b']:.0%})\n"
            f"→ {row['projected_winner']}"
        )

        plt.text(
            x / 4,
            y,
            text,
            ha="center",
            va="center",
            fontsize=8,
        )

plt.text(
    0.5,
    0.02,
    f"Single bracket projected champion: {projected_champion}",
    ha="center",
    fontsize=11,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "18_single_projected_knockout_bracket.png",
    dpi=180,
)
plt.show()


In [ ]:
report_zip = Path(
    "data/processed/"
    "18_post_group_stage_exact_knockout_forecast_report_bundle.zip"
)

with zipfile.ZipFile(
    report_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(OUTPUT_DIR))

print("Created:", report_zip)
